# Zoom Info

In [3]:
from seleniumbase import SB
import json
import time

url = "https://www.zoominfo.com/c/fever-tree/103761433"

with SB(uc=True, headless=False) as sb:
    sb.uc_open_with_reconnect(url, reconnect_time=4)
    time.sleep(3)

    script_html = sb.get_attribute('script#ng-state', 'innerHTML')
    raw_data = json.loads(script_html)
    data = raw_data.get("pageData", {})

# salva il JSON completo per consultazioni future
with open("fever_tree_full.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# estrai i campi principali
about = data.get("about", {})
overview = data.get("overview", {})

nome = data.get("name")
sede = data.get("address")
sito = about.get("displayLink")
telefono = about.get("phone")

# revenue è in migliaia di dollari, lo converto in milioni
revenue_raw = about.get("revenue")
revenue_milioni = int(revenue_raw) / 1000 if revenue_raw else None

dipendenti = data.get("numberOfEmployees")
ticker = overview.get("ticker")
sic_codes = overview.get("sic")
naics_codes = overview.get("naics")
anno_fondazione = data.get("foundingYear")

# industries è una lista di dict, prendo solo i nomi
industries_list = about.get("industries", [])
industries = [ind["name"] for ind in industries_list]

# tech stack - utile per la tua ricerca su ransomware
tech_used = data.get("techUsed", [])

# competitor
competitors_list = data.get("competitors", [])

print("Nome:", nome)
print("Sede:", sede)
print("Sito:", sito)
print("Telefono:", telefono)
print(f"Revenue: ${revenue_milioni}M" if revenue_milioni else "Revenue: N/A")
print("Dipendenti:", dipendenti)
print("Ticker:", ticker)
print("Anno fondazione:", anno_fondazione)
print("SIC:", sic_codes)
print("NAICS:", naics_codes)
print("Industries:", industries)
print("Numero tech usate:", len(tech_used))
print("Numero competitor:", len(competitors_list))

Nome: Fever-Tree
Sede: {'street': '186 -188 Shepherds Bush Rd', 'city': 'London', 'state': 'Greater London', 'country': 'United Kingdom', 'zip': 'W6 7NL'}
Sito: www.fever-tree.com
Telefono: +44 2073494922
Revenue: $459.965M
Dipendenti: 380
Ticker: AIM: FEVR
Anno fondazione: 2005
SIC: ['20', '208']
NAICS: ['31', '312']
Industries: ['Food & Beverage', 'Retail', 'Manufacturing', 'Grocery Retail']
Numero tech usate: 4
Numero competitor: 6


# Scare

In [1]:
from seleniumbase import SB
from pathlib import Path
import json
import time
import random
import csv

INPUT_CSV = "zoominfo_urls.csv"
OUTPUT_CSV = "zoominfo_data.csv"

# colonne del CSV di output
FIELDS = [
    "zoom_url", "company_id", "name", "full_name",
    "company_website", "founding_year", "is_public", "ticker",
    "num_employees", "revenue_millions_usd", "phone",
    "address_street", "address_city", "address_state",
    "address_country", "address_zip",
    "description", "industries", "sic_codes", "naics_codes",
    "tech_used", "competitors", "ceo_name", "num_executives",
    "total_funding", "scrape_status",
]


def parse_company(page_data, url):
    """Estrae i campi rilevanti dal JSON pageData."""
    about = page_data.get("about") or {}
    overview = page_data.get("overview") or {}
    address = page_data.get("address") or {}

    # revenue: in migliaia di dollari, lo converto in milioni
    revenue_raw = about.get("revenue")
    revenue_millions = None
    if revenue_raw and str(revenue_raw).isdigit():
        revenue_millions = int(revenue_raw) / 1000

    # industries: lista di dict -> stringa con ";"
    industries_list = about.get("industries") or []
    industry_names = [ind.get("name", "") for ind in industries_list]
    industries = "; ".join(n for n in industry_names if n)

    # tech stack: lista di dict
    tech_list = page_data.get("techUsed") or []
    tech_names = [t.get("name", "") for t in tech_list if isinstance(t, dict)]
    tech_used = "; ".join(n for n in tech_names if n)

    # competitor: lista di dict
    competitors_list = page_data.get("competitors") or []
    competitor_names = [c.get("name", "") for c in competitors_list if isinstance(c, dict)]
    competitors = "; ".join(n for n in competitor_names if n)

    # CEO
    ceo = page_data.get("ceo") or {}
    ceo_name = ceo.get("name") if isinstance(ceo, dict) else None

    # executives
    executives = page_data.get("executives") or []

    # codici SIC/NAICS
    sic = overview.get("sic") or []
    naics = overview.get("naics") or []

    return {
        "zoom_url": url,
        "company_id": page_data.get("companyId"),
        "name": page_data.get("name"),
        "full_name": page_data.get("fullName"),
        "company_website": about.get("displayLink"),
        "founding_year": page_data.get("foundingYear"),
        "is_public": page_data.get("isPublic"),
        "ticker": overview.get("ticker"),
        "num_employees": page_data.get("numberOfEmployees"),
        "revenue_millions_usd": revenue_millions,
        "phone": about.get("phone"),
        "address_street": address.get("street"),
        "address_city": address.get("city"),
        "address_state": address.get("state"),
        "address_country": address.get("country"),
        "address_zip": address.get("zip"),
        "description": page_data.get("description"),
        "industries": industries,
        "sic_codes": "; ".join(sic),
        "naics_codes": "; ".join(naics),
        "tech_used": tech_used,
        "competitors": competitors,
        "ceo_name": ceo_name,
        "num_executives": len(executives),
        "total_funding": page_data.get("totalFundingAmount"),
        "scrape_status": "ok",
    }


def load_urls_to_do():
    """Legge gli URL input e salta quelli già scrapati."""
    all_urls = []
    with open(INPUT_CSV) as f:
        reader = csv.DictReader(f)
        for row in reader:
            all_urls.append(row["zoom_url"].strip())

    done = set()
    if Path(OUTPUT_CSV).exists():
        with open(OUTPUT_CSV) as f:
            reader = csv.DictReader(f)
            for row in reader:
                done.add(row["zoom_url"])

    todo = [u for u in all_urls if u not in done]
    print(f"Totale: {len(all_urls)} | Già fatti: {len(done)} | Da fare: {len(todo)}")
    return todo


def write_row(row):
    """Appende una riga al CSV (crea header la prima volta)."""
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


# ============= MAIN =============
todo = load_urls_to_do()

with SB(uc=True, headless=False) as sb:
    for i, url in enumerate(todo, 1):
        print(f"\n[{i}/{len(todo)}] {url}")

        sb.uc_open_with_reconnect(url, reconnect_time=4)
        time.sleep(random.uniform(2, 4))

        # se il tag non c'è (CAPTCHA o blocco), salva un placeholder e va avanti
        if not sb.is_element_present('script#ng-state'):
            print("  -> tag ng-state non trovato (probabile blocco)")
            write_row({"zoom_url": url, "scrape_status": "blocked"})
            time.sleep(random.uniform(15, 25))
            continue

        script_html = sb.get_attribute('script#ng-state', 'innerHTML')
        raw_data = json.loads(script_html)
        page_data = raw_data.get("pageData") or {}

        if not page_data:
            print("  -> pageData vuoto")
            write_row({"zoom_url": url, "scrape_status": "empty"})
            continue

        row = parse_company(page_data, url)
        write_row(row)
        print(f"  -> OK: {row['name']} | revenue={row['revenue_millions_usd']}M | dipendenti={row['num_employees']}")

        # delay random per non farsi bannare
        time.sleep(random.uniform(5, 12))

print(f"\nFinito! Output: {OUTPUT_CSV}")

Totale: 376 | Già fatti: 29 | Da fare: 347

[1/347] https://www.zoominfo.com/c/ashley/566157328
  -> tag ng-state non trovato (probabile blocco)

[2/347] https://www.zoominfo.com/c/asi-cloud-bank/363494513
  -> OK: ASI Cloud Bank | revenue=4.258M | dipendenti=22

[3/347] https://www.zoominfo.com/c/ast-corp/11701126
  -> pageData vuoto

[4/347] https://www.zoominfo.com/c/auto-future-inc/351928241
  -> OK: Auto Future | revenue=14.862M | dipendenti=57

[5/347] https://www.zoominfo.com/c/ayesa/355599614
  -> OK: Ayesa | revenue=798.975M | dipendenti=13200

[6/347] https://www.zoominfo.com/c/barrack-rodos--bacine/3998523
  -> OK: Barrack Rodos & Bacine | revenue=15.93M | dipendenti=60

[7/347] https://www.zoominfo.com/c/bath-group-inc/29511555
  -> OK: Bath Group | revenue=11.487M | dipendenti=78

[8/347] https://www.zoominfo.com/c/bd-energy-systems-llc/356812163
  -> OK: BD Energy Systems | revenue=66.534M | dipendenti=60

[9/347] https://www.zoominfo.com/c/beko-technologies-ltd/347737721

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=147.0.7727.138)
Stacktrace:
0   uc_driver                           0x000000010292b028 uc_driver + 6823976
1   uc_driver                           0x00000001029226ba uc_driver + 6788794
2   uc_driver                           0x0000000102317a15 uc_driver + 453141
3   uc_driver                           0x00000001022ead58 uc_driver + 269656
4   uc_driver                           0x0000000102399d98 uc_driver + 986520
5   uc_driver                           0x00000001023a3531 uc_driver + 1025329
6   uc_driver                           0x000000010235e4b4 uc_driver + 742580
7   uc_driver                           0x000000010235f261 uc_driver + 746081
8   uc_driver                           0x00000001028e2da7 uc_driver + 6528423
9   uc_driver                           0x00000001028e71e6 uc_driver + 6545894
10  uc_driver                           0x00000